# Perceptual Decision-Making

**Author:** Teo Fantacci  
**Purpose:** Pedagogical companion to Froudist-Walsh et al. (in press, *Biol Psychiatry*) — illustrates a 3-population firing-rate winner-take-all (WTA) model of perceptual decision-making following Wong & Wang (2006). Corresponds to **Fig 3B** and **§3.1** of the review.

## How to cite
If you find this code useful for your research, please cite:
- The original paper: Wong K-F, Wang X-J (2006): A recurrent network mechanism of time integration in perceptual decisions. *J Neurosci* 26: 1314–1328.
- The review: Froudist-Walsh S, Ivanov TG, Magrou L, Cho H, Busch AN, Muller LE, et al. (in press): Mechanistic computational models of cognition across scales and species. *Biol Psychiatry*.

## Requirements
```bash
pip install numpy matplotlib
```
Runs in Google Colab (recommended) or locally with Jupyter (Python ≥ 3.8).  
No external data files required — all simulations are self-contained.

## Notes
All code is original. No third-party data or figures are included.
Simulation outputs are generated at runtime and not stored in the repository.


## Model overview

This notebook implements a **3-population firing-rate model** (two selective excitatory pools E1, E2 and one inhibitory pool I) that exhibits winner-take-all competition under symmetric input.

## Dynamics

**Notation:** $i \in \{1, 2\}$ indexes the two excitatory populations ($r_i = r_E$ via the f-I curve below); $I$ is the inhibitory pool; $k$ is any population. $r$ denotes firing rates (Hz), $S$ synaptic gating fractions, $J$ connectivity weights, $\tau$ time constants.

Each excitatory population $i$ carries two recurrent gating variables (slow NMDA, fast AMPA); the inhibitory pool has one (GABA):

$$
\frac{dS^{\text{NMDA}}_i}{dt} = -\frac{S^{\text{NMDA}}_i}{\tau_{\text{NMDA}}} + \gamma\,(1 - S^{\text{NMDA}}_i)\, r_i
\qquad
\frac{dS^{\text{AMPA}}_i}{dt} = -\frac{S^{\text{AMPA}}_i}{\tau_{\text{AMPA}}} + r_i
$$

$$
\frac{dS^{\text{GABA}}_I}{dt} = -\frac{S^{\text{GABA}}_I}{\tau_{\text{GABA}}} + \gamma_I\, r_I
$$

The total synaptic current to population $k$ sums recurrent contributions, background drive, the task stimulus, and noise:

$$
I_k = I_k^{\text{back}} + \sum_j J^{\text{NMDA}}_{kj}\, S^{\text{NMDA}}_j + \sum_j J^{\text{AMPA}}_{kj}\, S^{\text{AMPA}}_j - J^{\text{GABA}}_{kI}\, S^{\text{GABA}}_I + I_k^{\text{ext}}(t) + \eta_k(t)
$$

Firing rates follow from the f-I curves (Abbott–Chance for E, linear-threshold for I):

$$
r_E = \frac{aI - b}{1 - \exp\bigl(-d\,(aI - b)\bigr)}
\qquad
r_I = \max\bigl(c\, I - r_0,\, 0\bigr)
$$

Two ingredients drive the WTA competition: strong recurrent self-excitation on each E pool, and shared inhibition through the I pool. When E1 and E2 receive identical input, noise $\eta(t)$ alone breaks the symmetry — whichever pool randomly fires more recruits its own self-excitation faster and suppresses its competitor via the I pool.

## Reference and scope

The model is **inspired by** Wong & Wang (2006), *J. Neurosci.* 26(4):1314–1328, but it is **not a 1:1 replication** of either reduced model in that paper:

- The W&W 2006 *main* 2-variable model (Methods, p. 1317) has only E1 and E2; the inhibitory pool is **eliminated by linearization** and absorbed into the cross-coupling between E populations.
- The W&W 2006 *appendix* 2-variable model (p. 1327) is even simpler (no AMPA at recurrent synapses).
- The model here keeps the **inhibitory pool explicit** for pedagogical clarity. The connectivity weights are hand-tuned for this 3-population architecture.
- Each excitatory pool has **two recurrent gating variables**: NMDA (slow, τ=100 ms) and AMPA (fast, τ=2 ms). The inhibitory pool has GABA gating (τ=5 ms).
- E→I and I→E/I currents are NMDA/GABA only (no AMPA on E→I), as a simplification.

## What the model does

Two selective E populations receive the *same* stimulus amplitude. Symmetry-breaking by stochastic noise determines which population wins. The inhibitory pool provides shared recurrent inhibition.

## Noise options

The notebook supports two noise models, selectable in the parameter dict:

- **`"gaussian"`**: independent Gaussian sample per timestep added to total current. Simple but the effective amplitude depends on `dt`.
- **`"ou"`**: Ornstein–Uhlenbeck process filtered by τ_AMPA = 2 ms (as in Wong & Wang 2006). More biophysically grounded and `dt`-independent.


## Setup: Drive and results folder (Colab) / local fallback

If running in Google Colab, mount Drive and use a folder there to save results. If running locally (e.g. with `jupyter notebook`), fall back to the current working directory. **Edit `tutorial_path` to point to your own folder before running.**


In [ ]:
import os

# If running in Google Colab, optionally point this to your own Drive folder.
# Otherwise leave as './' — the notebook will work in the current directory.
tutorial_path = './'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Optional: uncomment and edit the line below to save to your Drive instead
    # tutorial_path = '/content/drive/MyDrive/your_folder_here/'
    print(f"Running in Colab. Working in: {tutorial_path}")
except (ImportError, ModuleNotFoundError):
    tutorial_path = os.getcwd()
    print(f"Running locally. Working in: {tutorial_path}")

saved_results = os.path.join(tutorial_path, 'saved_results')
os.makedirs(saved_results, exist_ok=True)
print(f"Results will be saved under: {saved_results}")


In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt


## Parameters

Only the parameters most likely to be varied are exposed here. All other constants (τ values, γ, background currents, stimulus amplitudes, f-I curve constants, noise amplitudes, integration step) are hard-coded inside the simulation function — see the function body if you need to inspect or change them.


In [ ]:
PARAMS = {
    # --- Time grid (seconds) ---
    "time": {
        "trial_length": 4.0,
        "stim_on": 1.0,
        "stim_off": 1.5,
    },

    # --- Connectivity (nA) ---
    # All weights are positive; sign of the synaptic effect is set inside
    # build_J() (I -> targets enters with a minus sign).
    # The recurrent E->E weights are split into NMDA (slow) and AMPA (fast)
    # contributions following W&W 2006 ratios: AMPA carries ~20% of the
    # sustained self-excitation and ~10% of the cross-coupling, but its
    # raw conductance is large because S_AMPA = tau_AMPA * r is small.
    "connectivity": {
        # E -> same E (recurrent self-excitation)
        "g_NMDA_self":  0.304,
        "g_AMPA_self":  0.886,
        # E -> other E (weak cross-excitation)
        "g_NMDA_cross": 0.009,
        "g_AMPA_cross": 0.012,
        # E -> I  (NMDA only, no AMPA on E->I in this model)
        "g_e_to_i":     0.250,
        # I -> E  (GABA, sign added in matrix)
        "g_i_to_e":     0.300,
        # I -> I  (GABA self-inhibition, sign added in matrix)
        "g_i_to_i":     0.200,
    },

    # --- Noise model ---
    # "gaussian": iid Gaussian per step added to total current (simple, dt-dependent)
    # "ou":       Ornstein-Uhlenbeck process filtered by tau_AMPA (W&W 2006 style)
    "noise_type": "ou",

    # --- Simulation control ---
    "simulation": {
        "n_runs": 6,
        "seed": 0,           # base seed for reproducibility; trial i uses seed+i
    },
}


## Building the connectivity matrices

The model has two recurrent gating channels for E populations (NMDA and AMPA), so we build two separate connectivity matrices: one acting on the NMDA gating vector `S_NMDA` and one acting on the AMPA gating vector `S_AMPA`. The GABA contribution from the I population enters both.

Each matrix has shape `[3, 3]` with rows = TO population and columns = FROM population, in order `[E1, E2, I]`.


In [ ]:
def build_J_matrices(conn):
    """Assemble the NMDA and AMPA connectivity matrices for [E1, E2, I].

    NMDA matrix carries: E->E recurrent (NMDA part), E->I, I->E/I (GABA).
    AMPA matrix carries: E->E recurrent (AMPA part) only. E->I has no AMPA
    here. The I column of the AMPA matrix is zero.

    Inhibitory entries (I -> targets) appear with a minus sign.
    """
    g_N_s  = conn["g_NMDA_self"]
    g_N_x  = conn["g_NMDA_cross"]
    g_A_s  = conn["g_AMPA_self"]
    g_A_x  = conn["g_AMPA_cross"]
    g_ei   = conn["g_e_to_i"]
    g_ie   = conn["g_i_to_e"]
    g_ii   = conn["g_i_to_i"]

    J_NMDA = np.array([
        # From E1   From E2    From I
        [g_N_s,     g_N_x,     -g_ie],   # To E1
        [g_N_x,     g_N_s,     -g_ie],   # To E2
        [g_ei,      g_ei,      -g_ii],   # To I
    ])

    J_AMPA = np.array([
        # From E1   From E2    From I
        [g_A_s,     g_A_x,     0.0],     # To E1
        [g_A_x,     g_A_s,     0.0],     # To E2
        [0.0,       0.0,       0.0],     # To I (no AMPA on E->I in this model)
    ])

    return J_NMDA, J_AMPA


## f-I (input → firing rate) curves

Excitatory: Abbott–Chance form fitted to LIF in Wong & Wang 2006 (Eq. 2, p. 1315), with parameters `a=310 (V·nC)⁻¹`, `b=125 Hz`, `d=0.16 s`.

Inhibitory: linear-threshold approximation `r = max(c·I − r₀, 0)` with `c=615`, `r₀=177 Hz`. (See model overview for caveats.)


In [ ]:
def fI_excitatory(I):
    """Abbott-Chance input-output function for E populations (W&W 2006 p. 1315)."""
    a, b, d = 310.0, 125.0, 0.16
    x = a * I - b
    return x / (1.0 - np.exp(-d * x) + 1e-6) # + 1e-6 to prevent divide-by-zero when x is very small


def fI_inhibitory(I):
    """Linear-threshold rate for the I population."""
    c, r_0 = 615.0, 177.0
    return np.maximum(c * I - r_0, 0.0)


## Core simulation

`run_simulation(params, seed=...)` runs one trial and returns a result dict containing the time grid, firing-rate trace `R` (shape `[3, n_steps]`), the gating traces, and the params used.

Hard-coded constants live in this function:

- `dt = 0.5 ms` (integration step)
- `tau = [100, 100, 5] ms` (NMDA for E, GABA for I), `tau_AMPA = 2 ms`, `tau_OU = 2 ms`
- `gamma = [0.641, 0.641, 1.0]` (NMDA saturation factor; I and AMPA have no saturation)
- `I_back = [0.310, 0.310, 0.227] nA`, `stim_amp = [0.050, 0.050, 0.0] nA`
- noise amplitude `sigma_E = 0.005 nA` (Gaussian) or `sigma_E_ou = 0.007 nA` (OU, matching W&W 2006)


In [ ]:
def run_simulation(params, seed=None):
    """Run one trial of the 3-population WTA network with NMDA + AMPA gating.

    Parameters
    ----------
    params : dict
        Slim parameter dict (see PARAMS template).
    seed : int or None
        If given, seeds a local numpy RNG for reproducibility. Falls back to
        params["simulation"]["seed"] if None.

    Returns
    -------
    dict with keys: 'time', 'R' (3 x n_steps firing rates),
    'S_NMDA', 'S_AMPA' (3 x n_steps; AMPA col for I is unused/zero),
    'S_GABA' (3 x n_steps; only the I row is meaningful),
    'stim_on', 'stim_off', 'params'.
    """
    # --- Hard-coded constants ---
    dt = 0.0005                                # integration step (s)
    tau = np.array([0.100, 0.100, 0.005])      # NMDA(E1, E2), GABA(I)
    tau_AMPA = 0.002                           # AMPA gating time constant (s)
    tau_OU = 0.002                             # OU noise time constant (s)
    gamma = np.array([0.641, 0.641, 1.000])    # NMDA saturation factor; GABA has no saturation
    I_back = np.array([0.310, 0.310, 0.227])   # background current per population (nA)
    stim_amp = np.array([0.050, 0.050, 0.0])   # task stimulus per population (nA)
    sigma_gauss = 0.005                        # Gaussian noise amplitude (nA, applied to E only)
    sigma_ou = 0.007                           # OU noise amplitude (nA, W&W 2006 style)
    S_init = 0.05                              # initial gating value

    # --- Unpack slim dict ---
    p_t = params["time"]
    noise_type = params["noise_type"]
    if seed is None:
        seed = params["simulation"].get("seed", None)
    rng = np.random.default_rng(seed)

    # --- Time grid ---
    time = np.arange(0.0, p_t["trial_length"], dt)
    n_steps = len(time)
    stim_on, stim_off = p_t["stim_on"], p_t["stim_off"]

    # --- Connectivity ---
    J_NMDA, J_AMPA = build_J_matrices(params["connectivity"])

    # --- State arrays ---
    S_NMDA = np.zeros((3, n_steps))     # NMDA gating per population (E1, E2 used)
    S_AMPA = np.zeros((3, n_steps))     # AMPA gating per population (E1, E2 used)
    S_GABA = np.zeros((3, n_steps))     # GABA gating (only I row used)
    R      = np.zeros((3, n_steps))     # firing rates

    S_NMDA[0:2, 0] = S_init
    S_AMPA[0:2, 0] = S_init
    S_GABA[2, 0]   = S_init

    # OU noise state (one process per E population). Initialize at zero.
    I_noise = np.zeros(2)

    # --- Euler integration ---
    for t in range(n_steps - 1):
        # External (task) input
        I_ext = stim_amp if (stim_on <= time[t] <= stim_off) else np.zeros(3)

        # --- Noise ---
        # Only E populations receive noise; I is noiseless here.
        if noise_type == "gaussian":
            # iid Gaussian per step (dt-dependent effective amplitude).
            noise_vec = np.zeros(3)
            noise_vec[0:2] = rng.standard_normal(2) * sigma_gauss
        elif noise_type == "ou":
            # OU update: dI = -(I/tau_OU) dt + sigma * sqrt(dt/tau_OU) * xi
            # (discrete OU as in W&W 2006; keeps amplitude dt-independent.)
            xi = rng.standard_normal(2)
            I_noise = (I_noise
                       - (I_noise / tau_OU) * dt
                       + sigma_ou * np.sqrt(dt / tau_OU) * xi)
            noise_vec = np.zeros(3)
            noise_vec[0:2] = I_noise
        else:
            raise ValueError(f"Unknown noise_type: {noise_type!r}. Use 'gaussian' or 'ou'.")

        # --- Total synaptic current to each population ---
        # NMDA contributions use S_NMDA (E columns) and S_GABA (I column).
        # AMPA contributions use S_AMPA (E columns only; I column is zero).
        # We assemble a single "S vector" for each matrix:
        S_for_NMDA = np.array([S_NMDA[0, t], S_NMDA[1, t], S_GABA[2, t]])
        S_for_AMPA = np.array([S_AMPA[0, t], S_AMPA[1, t], 0.0])

        I_tot = I_back + J_NMDA @ S_for_NMDA + J_AMPA @ S_for_AMPA + I_ext + noise_vec

        # --- Firing rates ---
        R[0, t] = fI_excitatory(I_tot[0])
        R[1, t] = fI_excitatory(I_tot[1])
        R[2, t] = fI_inhibitory(I_tot[2])

        # --- Gating dynamics ---
        # NMDA (E1, E2): saturating term (1 - S_NMDA), driven by population rate
        dS_NMDA = (-S_NMDA[0:2, t] / tau[0:2]
                   + gamma[0:2] * (1.0 - S_NMDA[0:2, t]) * R[0:2, t])
        # AMPA (E1, E2): no saturation, fast time constant
        dS_AMPA = -S_AMPA[0:2, t] / tau_AMPA + R[0:2, t]
        # GABA (I): no saturation
        dS_GABA = -S_GABA[2, t] / tau[2] + gamma[2] * R[2, t]

        S_NMDA[0:2, t+1] = S_NMDA[0:2, t] + dS_NMDA * dt
        S_AMPA[0:2, t+1] = S_AMPA[0:2, t] + dS_AMPA * dt
        S_GABA[2, t+1]   = S_GABA[2, t]   + dS_GABA * dt

    # Boundary fill of the final firing-rate column (loop ends at n_steps-1).
    I_ext_last = stim_amp if (stim_on <= time[-1] <= stim_off) else np.zeros(3)
    S_for_NMDA_last = np.array([S_NMDA[0, -1], S_NMDA[1, -1], S_GABA[2, -1]])
    S_for_AMPA_last = np.array([S_AMPA[0, -1], S_AMPA[1, -1], 0.0])
    I_tot_last = I_back + J_NMDA @ S_for_NMDA_last + J_AMPA @ S_for_AMPA_last + I_ext_last
    R[0, -1] = fI_excitatory(I_tot_last[0])
    R[1, -1] = fI_excitatory(I_tot_last[1])
    R[2, -1] = fI_inhibitory(I_tot_last[2])

    return {
        "time": time,
        "R": R,
        "S_NMDA": S_NMDA,
        "S_AMPA": S_AMPA,
        "S_GABA": S_GABA,
        "stim_on": stim_on,
        "stim_off": stim_off,
        "params": params,
    }


## Plotting

`plot_trial(result, ax)` draws one trial on a given matplotlib axis. `run_and_plot(params, n_runs)` orchestrates a single figure with one subplot per trial.


In [ ]:
# Okabe-Ito colorblind-friendly palette
_COLORS = {"E1": "#0072B2", "E2": "#D55E00", "I": "black"}


def plot_trial(result, ax):
    """Plot firing rates of one trial on the given axis."""
    time = result["time"]
    R = result["R"]
    ax.plot(time, R[0], label="E1", color=_COLORS["E1"], lw=2.5, zorder=1)
    ax.plot(time, R[1], label="E2", color=_COLORS["E2"], lw=2.5, zorder=1)
    ax.plot(time, R[2], label="I",  color=_COLORS["I"],  lw=1.2, ls="--", zorder=5)
    ax.axvspan(result["stim_on"], result["stim_off"],
               color="grey", alpha=0.15, label="Stimulus", zorder=0)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Firing rate (Hz)")
    ax.grid(True, ls=":", alpha=0.6)


def _winner_label(R):
    """Return 'E1', 'E2', or 'Tie' based on the final firing rates."""
    if R[0, -1] > R[1, -1]:
        return "E1"
    if R[1, -1] > R[0, -1]:
        return "E2"
    return "Tie"


def run_and_plot(params, savedir=None, plot_ext=".png"):
    """Run n_runs trials (from params['simulation']['n_runs']) and plot in a single figure.

    n_runs == 1 -> single panel.
    n_runs  > 1 -> grid of subplots (max 3 columns).

    If savedir is given, the figure is saved there.
    Returns None to avoid the Jupyter/Colab duplicate-figure issue.
    """
    n_runs = params["simulation"]["n_runs"]
    base_seed = params["simulation"].get("seed", None)

    if n_runs == 1:
        fig, ax = plt.subplots(figsize=(10, 6))
        result = run_simulation(params, seed=base_seed)
        plot_trial(result, ax)
        ax.set_title(f"WTA dynamics (winner: {_winner_label(result['R'])})")
        ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.0))
        fname = f"wta_single_run{plot_ext}"

    else:
        cols = min(n_runs, 3)
        rows = math.ceil(n_runs / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows),
                                 sharex=True, sharey=True)
        axes = np.atleast_1d(axes).flatten()

        for i in range(n_runs):
            # Per-trial seed: distinct but reproducible if base_seed given.
            trial_seed = (base_seed + i) if base_seed is not None else None
            result = run_simulation(params, seed=trial_seed)
            plot_trial(result, axes[i])
            axes[i].set_title(f"Trial {i + 1} (winner: {_winner_label(result['R'])})")

        for j in range(n_runs, len(axes)):
            axes[j].set_visible(False)

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="center right",
                   bbox_to_anchor=(1.08, 0.5))
        fig.suptitle(f"WTA network: {n_runs} independent trials "
                     f"(noise: {params['noise_type']})", y=1.02)
        fname = f"wta_{n_runs}_runs_{params['noise_type']}{plot_ext}"

    plt.tight_layout()

    if savedir is not None:
        os.makedirs(savedir, exist_ok=True)
        full = os.path.join(savedir, fname)
        plt.savefig(full, dpi=300, bbox_inches="tight")
        print(f"Saved: {full}")

    plt.show()
    plt.close(fig)   # prevents the duplicate-figure issue in Colab/Jupyter
    return None


## Run

Edit `PARAMS` above (e.g. `PARAMS["noise_type"] = "gaussian"` or `PARAMS["simulation"]["n_runs"] = 1`) and re-run.


In [ ]:
wta_savedir = os.path.join(saved_results, 'winner_take_all')
run_and_plot(PARAMS, savedir=wta_savedir)


## Biological context

### What the model represents

The WTA model is a mean-field reduction of a spiking network in the lateral intraparietal area (LIP) or middle temporal area (MT) — brain regions where neurons signal perceptual decisions about moving stimuli. In the classic Roitman & Shadlen (2002) task, a monkey watches random dots moving left or right and must report the direction. Two populations of LIP neurons — one tuned to "leftward motion," the other to "rightward motion" — accumulate sensory evidence and compete for the decision via shared feedback inhibition.

### Why mean-field?

Each `E1`/`E2`/`I` unit in our 3-equation model stands in for a population of hundreds to thousands of real neurons that share similar stimulus selectivity. The synaptic gating variable `S_NMDA` represents the *fraction of NMDA receptors open* at the population level, not at single neurons. This averaging lets us collapse a network of ~2000 spiking neurons (as in Wang 2002) into three ODEs that we can analyze and simulate in seconds. The price: we lose spike-timing information (synchrony, gamma oscillations, individual spike patterns).

### The role of each time constant

- **NMDA (τ = 100 ms)**: slow glutamatergic excitation. This is the *integration timescale* that lets the network accumulate sensory evidence over hundreds of milliseconds. Without NMDA's slow kinetics, perceptual decisions couldn't take the seconds they do in real life.
- **AMPA (τ = 2 ms)**: fast glutamatergic excitation. Responsible for the *rising-rate transient* during stimulus presentation — when the cued population spikes hard, AMPA tracks it instantly. In LIP recordings, this corresponds to the fast initial rise of firing rates when a stimulus arrives.
- **GABA (τ = 5 ms, via the I pool)**: fast inhibition. Balances the fast excitatory transients so the network doesn't run away.

### The winner-take-all mechanism

Two ingredients:
1. **Self-excitation**: each E population has strong recurrent coupling to itself (`g_e_self`). Once active, it stays active.
2. **Shared inhibition**: both E populations drive the same I pool, which in turn inhibits both. If E1 fires more than E2, I suppresses E2 more than E1 (because E1's own self-excitation keeps it going while E2 loses against inhibition). The asymmetry amplifies.

This is the same mechanism the brain uses for many "pick one of N options" problems: visual attention, saccade target selection, hand choice in reaching. The small noise we inject is what breaks the symmetry on ambiguous trials — behaviorally, this corresponds to why human or monkey choices on 50/50 stimuli are not deterministic.

### What this model can and cannot tell you

It *can* tell you:
- How decision accuracy depends on stimulus strength (harder evidence → more variable decisions)
- How reaction time depends on recurrent excitation strength (stronger recurrence → faster buildup → faster decisions)
- That two-alternative choice is a bifurcation: there is no "partial winner" equilibrium in the bistable regime

It *cannot* tell you:
- Anything about sensory feature selectivity (we assume E1 and E2 are already tuned)
- Anything about learning (weights are fixed)
- Anything about multi-alternative choice with N > 2 (requires the WM-style generalization)
